# Chapter 13: Digital Forensics

> "The goal of digital forensics is to recover and analyze evidence in a manner that preserves
> its integrity so that it can be used in a legal proceeding." standard forensic doctrine

---

## Learning Objectives

After completing this chapter, you will be able to:

1. Explain the principles of digital forensics and the chain of custody.
2. Describe the forensic acquisition process and the difference between logical and physical images.
3. Apply hash verification to prove evidence integrity.
4. Describe file system artefacts including deleted files, slack space, and timestamps.
5. Explain memory forensics and what can be recovered from a RAM dump.
6. Describe network forensics and the value of PCAP analysis.
7. Explain log analysis in a forensic context and the significance of timestamp correlation.
8. Recognize anti-forensic techniques and their countermeasures.

## Key Terms

- **Order of volatility**: collect most-ephemeral evidence first (registers->RAM->network->disk->backups).
- **Chain of custody**: documented unbroken record of evidence handling for admissibility.
- **Write blocker / forensic image / hashing**: tools and steps that preserve and prove evidence integrity.
- **Daubert standard / Federal Rules of Evidence**: legal tests for admissible expert/forensic methods.

- **AI evidence triage**: machine-learning prioritization of which artifacts to analyze first.
- **IoT forensics**: forensics of smart-device, network, and cloud evidence.
- **Explainable AI (XAI)**: interpretable model outputs needed for admissible AI-assisted forensics.
- **Cyberattack attribution detection (CAD)**: identifying an attack's source/characteristics from forensic traces.

- **Chain of custody**: documented record of who had custody of evidence and when.
- **Forensic image**: a bit-for-bit copy of a storage device including unallocated space.
- **Write blocker**: hardware or software preventing any write to the source device during acquisition.
- **Hash verification**: computing a cryptographic hash of the image to prove it is unchanged.
- **Deleted file recovery**: recovering files whose directory entries have been removed but data not overwritten.
- **Slack space**: unused space in a file's last allocated cluster, potentially containing old data.
- **MFT**: Master File Table; NTFS structure recording metadata for every file.
- **Memory forensics**: acquiring and analyzing the contents of RAM.
- **Volatility**: open-source memory forensics framework.
- **PCAP**: packet capture; a file recording raw network traffic.
- **Artefact**: forensically significant data recovered from a system.
- **Anti-forensics**: techniques used to destroy or hide digital evidence.

---

## 13.1 Forensic Principles

### The Locard Exchange Principle

Locard's exchange principle from physical forensics holds that every contact leaves a trace.
In digital environments: every user action creates artefacts. Login events appear in logs, web
browsing creates browser history and DNS cache entries, file access updates timestamps and
prefetch records, and even deleted files often leave traces in unallocated space, the MFT, or
log files. The forensic investigator's task is to find, preserve, and interpret these traces.

### Forensic Soundness

Forensic soundness requires that the investigative process does not alter the evidence being
examined. Working from a verified copy (forensic image) of the original device, using a write
blocker during acquisition, computing hashes before and after, and maintaining a chain of custody
record are the foundational practices that allow a court to trust the evidence presented.

#### Chain of Custody

The chain of custody is a chronological record of every person who had control of the evidence
and every transfer between custodians. A break in the chain (unrecorded transfer, missing
signature) may allow a defense to argue that evidence was tampered with. For corporate
investigations that may lead to litigation or criminal referral, forensic investigators must
follow the same standards as law enforcement.

---


<!--CH13PROC-->
### The Forensic Process, Order of Volatility, and Chain of Custody

The principles above become a disciplined process whose every step must withstand legal scrutiny. The widely
used phases (NIST SP 800-86 and the ACPO model) are **identification, preservation, collection/acquisition,
examination, analysis, and reporting**. Two rules govern collection. First, the **order of volatility**:
collect the most ephemeral evidence first, CPU registers and cache, then RAM and running state, then network
connections, then disk, then archival media and backups, because volatile data vanishes on power loss. Second,
**preserve the original**: acquire a bit-for-bit **forensic image** using a **write blocker** so the source is
never altered, and **hash** both source and image (Chapter 2) to prove they match, repeating the hash later to
prove the evidence is unchanged (tamper-evidence in action).

Underpinning everything is the **chain of custody**: a documented, unbroken record of who handled the evidence,
when, why, and how, from seizure to courtroom. A gap in that chain, or a missing hash, can render even
damning evidence inadmissible. The four ACPO principles capture the ethic: do not change the data; if you must
access original data, be competent and explain why; keep an audit trail that a third party could reproduce; and
the case lead is responsible for compliance. This rigor is exactly what separates forensics from ordinary
incident analysis (Chapter 14), and it is the reason the AI-assisted methods earlier in this chapter must
remain explainable and human-validated.

## 13.2 Evidence Acquisition

### Write Blockers

A write blocker is inserted between the source device and the forensic workstation to prevent
any accidental write. Hardware write blockers (Tableau, WiebeTech) are preferred in legal
investigations because they are transparent to the OS and cannot be subverted by software. A
software write blocker (registry key in Windows, `blockdev --setro` in Linux) is acceptable in
internal investigations but may not withstand legal scrutiny.

### Forensic Imaging

A forensic image is a bit-for-bit copy of the entire device including allocated files, deleted
files, unallocated space, and slack space. `dd`, `dcfldd` (with hashing), and FTK Imager are
common tools. The image is usually stored in E01 (Expert Witness Format) which includes built-in
hash verification and compression.

#### Logical Versus Physical Acquisition

A logical acquisition copies only the file system's visible files and folders: faster and smaller
but misses deleted files and unallocated space. A physical (bit-level) acquisition copies the
entire device: slower but preserves all recoverable artefacts. For live systems, a cloud snapshot
or VM snapshot is an alternative when full physical imaging is impractical.

### Hash Verification

After acquisition, a hash (SHA-256) of the image is computed and recorded. At any later point,
re-hashing the image confirms it has not been modified. If the image hash matches the original,
the evidence is provably unchanged. SHA-256 is preferred over MD5 (which is cryptographically
broken) for new investigations, though MD5 is still widely used in practice due to tool defaults.

---

## 13.3 File System Forensics

### NTFS Artefacts

#### The Master File Table

The NTFS MFT records metadata for every file and directory: filename, timestamps (created, modified,
accessed, MFT-changed), size, and a reference to the file's data clusters. When a file is deleted,
its MFT entry is marked as available but is not immediately overwritten. The entry may remain
intact for a long time on a lightly-used volume, allowing filename and timestamp recovery even when
the file data is gone.

#### Timestamps and Timestomping

NTFS maintains four timestamps per file (MACB: Modified, Accessed, Changed, Born). Timestamp
correlation between MFT, log files, and the $LogFile/$UsnJrnl change journal allows an investigator
to reconstruct a precise timeline. Anti-forensic tools can alter timestamps (timestomping) but
often fail to update all four NTFS timestamps consistently, leaving detectable anomalies.

### Deleted Files and Unallocated Space

When a file is deleted, its directory entry is marked as available and its clusters are returned
to the free-space bitmap, but the data is not overwritten. Data recovery tools (Autopsy, FTK,
PhotoRec) scan unallocated clusters for known file signatures (magic bytes) to carve deleted files.
Solid-state drives with TRIM enabled may zero deleted blocks immediately, reducing recovery prospects.

---


### Disk, Memory, and Mobile Forensics in Practice

With the process fixed, the practical craft spans several evidence types. **Disk and file-system forensics**
recovers far more than visible files: a forensic image is searched for **deleted files** (whose directory entry
is removed but whose data persists until overwritten), **file carving** reconstructs files from raw bytes by
header/footer signatures, and **slack space** and unallocated areas hide data. Investigators build a
**timeline** from filesystem timestamps (the NTFS Master File Table, the `$MFT`) and from the Windows
**registry**, browser history, and event logs to reconstruct user activity. Tools include `dd` and **FTK
Imager** for imaging, **Autopsy/The Sleuth Kit** for analysis, and `bulk_extractor` for carving.

**Memory forensics** analyzes a RAM capture with frameworks such as **Volatility** to recover running
processes, network connections, injected code, and encryption keys, evidence that never touches disk and is the
only way to catch fileless malware (Chapter 15). **Mobile and cloud forensics** extend the discipline to phones
(logical and physical extractions, often via tools like Cellebrite) and to data held by cloud providers (legal
process, API acquisition), which is increasingly where evidence lives. A short worked example ties it together.

```{admonition} Worked Example: verifying and timelining an image
:class: note
After seizing a laptop, an examiner (1) attaches the drive through a hardware **write blocker**; (2) images it
with `dd` or FTK Imager and records `sha256sum` of source and image, confirming they match; (3) loads the image
in Autopsy, recovers deleted files by carving, and exports the `$MFT`; (4) builds a **super-timeline** (e.g.,
with Plaso/log2timeline) merging filesystem, registry, and log timestamps; (5) writes findings with each
artifact's hash and the chain-of-custody record. If the laptop was still powered on at seizure, a RAM capture
(for Volatility) would be taken *first*, per the order of volatility.
```

## 13.4 Memory Forensics

### Why Memory Matters

RAM contains information that is never written to disk: decrypted encryption keys, running process
lists, network connections, command history, clipboard contents, and injected shellcode. An attacker
using fileless malware may leave almost no disk artefacts while leaving extensive memory artefacts.
Memory acquisition must occur before the system is powered off; shutdown destroys volatile evidence.

### Acquiring Memory

On Windows, tools including Magnet RAM Capture and WinPmem produce a memory dump. On Linux,
`/dev/mem` (if accessible), `LiME` (Loadable Kernel Module), and hypervisor snapshots provide
memory. Live acquisition from a running system is preferable to cold-boot attacks (briefly cooling
DRAM to slow decay and transplanting to another machine) which are physically invasive.

### Analyzing Memory with Volatility

The Volatility framework analyses memory dumps across Windows, Linux, and macOS. Key plugins:

| Plugin | Information extracted |
|---|---|
| `pslist / pstree` | Running processes and their hierarchy |
| `dlllist` | DLLs loaded into a process |
| `netscan` | Network connections and listening sockets |
| `cmdline` | Command-line arguments for each process |
| `malfind` | Memory regions with PAGE_EXECUTE and suspicious content |
| `hivelist / printkey` | Registry hives and specific key values |
| `filescan` | File handles open in memory |

---

## 13.5 Network Forensics

### PCAP Analysis

Network packet captures record the complete conversation between hosts. Wireshark and Zeek analyze
PCAPs. Investigative questions: Which hosts communicated with the C2 IP? Were credentials transmitted
in the clear? What files were transferred? What DNS queries preceded the attack?

#### Network Evidence Sources

- **Firewall logs**: connection records with source/destination, port, bytes, and action.
- **Proxy logs**: full URL and user-agent for web traffic.
- **DNS logs**: all queries and responses, with timestamps.
- **NetFlow**: flow-level summaries (no payload) for high-volume environments.
- **Full packet capture**: complete payload; highest fidelity but large storage requirement.

---

## 13.6 Anti-Forensics

Anti-forensic techniques attempt to prevent, delay, or mislead forensic investigation. Common techniques:

- **Secure file deletion**: overwriting data before deletion (DoD 5220.22-M wiping) prevents carving.
- **Encryption**: encrypted volumes require the key; without it, content is inaccessible.
- **Timestomping**: modifying file timestamps to confuse timeline analysis.
- **Log deletion**: clearing Windows Event Logs, clearing bash history, or disabling logging.
- **Steganography**: hiding data in innocuous files to exfiltrate without detection.
- **Fileless malware**: executing entirely in memory via PowerShell or WMI to avoid disk artefacts.

### Countermeasures

Centralized, append-only log storage (SIEM) makes local log deletion ineffective. Immutable cloud
logging services (AWS CloudTrail with Object Lock) survive even a full EC2 instance compromise.
Memory acquisition before shutdown captures fileless malware. Monitoring for log-clearing events
(Windows Event ID 1102 for Security log cleared) provides an alert when log deletion is attempted.

---



<!--AIFOR-->
## 13.7 Artificial Intelligence in Digital Evidence Triage

The forensic process described so far assumes a human examiner can review the evidence, but modern cases bury
investigators under terabytes from disks, phones, cloud accounts, and IoT devices, which is why **AI-driven
evidence triage** has become essential. Triage is the early-stage prioritization of which artifacts deserve
deep analysis, and where traditional triage is static and rule-based, AI brings machine learning, deep
learning, and neural networks to automate **pattern recognition, anomaly detection, and artifact
classification** (as surveyed by Zamil and Khan, 2025). Natural-language processing scans emails, chat logs,
and documents to surface relevant material; image classifiers flag illicit media; and models learn to separate
the forensically interesting from the mundane far faster than manual review.

The benefits are real, but so are the limits, and a forensic examiner must understand both. AI triage helps
with data complexity, scale, and even anti-forensic countermeasures, but the recurring challenges are
**limited and biased training datasets**, **legal admissibility**, and **explainability**, an opaque model
that flags a file but cannot justify why is a liability in court. This is why the field is converging on
**explainable AI (XAI)** for forensics: a model's output must be interpretable and defensible under the
evidentiary standards (Daubert/Frye, Chapter 18) that govern expert testimony. AI accelerates the examiner; it
does not replace the chain of custody, validation, and human judgment that make evidence admissible.


## 13.8 IoT Forensics and Explainable AI

The Internet of Things multiplies both the sources of evidence and the difficulty of collecting it. **IoT
forensics** extends digital forensics to the data on smart devices, their networks, and the cloud services
behind them, and it is forensically valuable because IoT devices are now both *targets* and *witnesses*: a
breach may run through them (DDoS, ransomware, data theft), and their sensor and log data can reconstruct what
happened (Gopinath et al., 2023). The challenges are acute: devices are heterogeneous and proprietary, storage
is tiny and volatile, data lives partly in the cloud, and there are few standards, so investigators rely on
general tools, **Autopsy**, **FTK Imager**, and **Wireshark** (Chapters 3 and 8), adapted to device images and
network captures, walking the standard phases (identification, preservation, acquisition, analysis,
presentation) under chain of custody even in "data stolen" or "data deleted" scenarios.

AI is reshaping this subfield too. The **IoT-CAD** work (Mohamed et al., 2025) builds a comprehensive IoT
forensics dataset for **Cyberattack Attribution Detection**, collecting traces from Windows and Linux hosts
(memory, disk, processes, system calls, and network traffic) across realistic IoT attack scenarios, then
evaluating deep-learning attribution under both **centralized** and **federated learning** (the
privacy-preserving paradigm of Chapter 17) with **explainable-AI** techniques and network forensics. The
lesson, consistent with the triage discussion above, is that attribution and triage increasingly depend on
high-quality datasets and on explanations an investigator can defend, which is why XAI is becoming a forensic
requirement rather than a nicety.


## 13.9 AI-Driven Cybercrime Analytics and Attribution

Beyond triage and IoT, AI is changing how investigators detect and attribute cybercrime at scale, the meeting
point of forensics (this chapter), detection (Chapter 12), and threat intelligence. As a concrete example,
Djenna and colleagues (2023) combine an **unsupervised LSTM** (to model normal sequential behavior) with a
**supervised CNN** (to classify) for early detection of **botnet** attacks, evaluated on the standard
**CTU-13** and **IoT-23** datasets and reporting very high accuracy (over 98.7%) at a low false-positive rate
(around 0.04%). The forensic relevance is twofold: such models triage which hosts in a captured dataset are
compromised, and they enrich **cyber threat intelligence** by recognizing emerging botnet families, feeding
both the investigation and the defense.

```{admonition} Worked Example: AI-assisted triage on a seized dataset
:class: note
Suppose an investigator seizes a 4 TB image plus a week of network captures from a small IoT-enabled office.
A defensible AI-assisted workflow is: (1) hash and image the evidence (preserve originals, chain of custody);
(2) run NLP keyword and entity models over documents/email to rank likely-relevant items; (3) apply an
LSTM/CNN flow classifier (as above) to the captures to flag hosts showing botnet/C2 behavior; (4) for each
AI-flagged item, record the model's **explanation** (which features drove the score) so it can be defended;
(5) hand the prioritized shortlist to a human examiner for full manual analysis and reporting. The AI changes
*the order and speed* of review, not the requirement that conclusions rest on validated, explainable, manually
confirmed evidence.
```

```{admonition} Knowledge Check
:class: tip
1. Why is explainability (XAI) becoming a requirement, not a luxury, in AI-assisted forensics?
2. What makes IoT forensics harder than traditional disk forensics, and name two tools used in practice.
3. In the LSTM+CNN botnet example, what is each component responsible for?

*Answers:* (1) Forensic conclusions must be admissible and defensible under expert-evidence standards
(Chapter 18); an unexplained model output cannot be justified in court, so its reasoning must be interpretable.
(2) IoT devices are heterogeneous and proprietary with tiny, volatile storage and cloud-resident data and few
standards; practitioners use Autopsy, FTK Imager, and Wireshark. (3) The unsupervised LSTM models normal
sequential behavior to spot deviations, and the supervised CNN classifies traffic as botnet or benign.
```


## 13.10 Legal Admissibility and Reporting

Forensic work exists to support legal or organizational decisions, so admissibility is not an afterthought but
a design constraint. In U.S. courts, expert testimony and scientific methods are judged under the **Daubert**
standard (is the technique testable, peer-reviewed, with a known error rate, and generally accepted?) and the
**Federal Rules of Evidence**, while authenticity (the evidence is what it claims to be) and the
**chain of custody** are threshold requirements (these connect to the privacy-and-law material of Chapter 18).
Practically, this means using **validated, reproducible tools and methods**, documenting every step so an
opposing expert could repeat it, retaining hashes to prove integrity, and writing a clear report that separates
fact from interpretation and states the limits of the findings. The rise of AI-assisted triage (earlier in this
chapter) heightens these demands: an automated result must be explainable and independently verifiable to meet
the same bar. The examiner's credibility, like the evidence's, rests on rigor and transparency.

```{admonition} Knowledge Check
:class: tip
1. State the order of volatility and why it dictates collection order.
2. What is the role of a write blocker and a hash in preserving evidence integrity?
3. Why does the Daubert standard matter for both traditional and AI-assisted forensic methods?

*Answers:* (1) Collect most-volatile first, registers/cache, RAM, network state, disk, backups, because
volatile data is lost on power-off and cannot be recovered later. (2) A write blocker prevents any change to the
original media during acquisition; hashing source and image (and re-hashing later) proves the copy is identical
and unaltered. (3) Daubert requires methods to be testable, peer-reviewed, with known error rates, and
accepted; AI methods must additionally be explainable and reproducible to satisfy it, or their output is
inadmissible.
```


## Why This Matters

Digital forensics is both a technical discipline and a legal process. Evidence that is improperly
collected or handled cannot be used in court and may allow a perpetrator to avoid consequences.
For corporate investigations, improperly handled forensic evidence can also expose the organization
to civil liability. Technical staff who understand forensic principles make better first responders:
they preserve volatile evidence, avoid contaminating the scene, and document their actions in a way
that supports subsequent investigation.

---

## News in Focus: When Digital Forensic Evidence Decides a Case
Several high-profile criminal prosecutions have turned on digital forensic evidence: recovered
deleted chat logs, memory dumps containing encryption keys, and network captures proving communication
with known malicious infrastructure. Equally, cases have been dismissed or weakened because
investigators failed to maintain chain of custody, used non-forensic acquisition methods, or
could not prove that evidence had not been altered. The technical quality of forensic work directly
determines whether justice is served.

---


In [1]:
# Chapter 13 -- Hash verification, file timestamp analysis, and artefact recovery simulation
import hashlib, os, time
from datetime import datetime
from io import BytesIO
from IPython.display import display, Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Hash verification demo ────────────────────────────────────────────────────
def sha256(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

original_image = b"FORENSIC_IMAGE_" + b"A" * 1000
img_hash = sha256(original_image)
print("=== Evidence Integrity Verification ===")
print(f"  Original image SHA-256 : {img_hash}")

verified = sha256(original_image)
print(f"  Re-verified SHA-256    : {verified}")
print(f"  Integrity check        : {'PASSED' if verified == img_hash else 'FAILED - TAMPERED!'}")

tampered = original_image[:-1] + b"B"
print(f"  Tampered image SHA-256 : {sha256(tampered)}")
print(f"  Tampered check         : {'PASSED' if sha256(tampered)==img_hash else 'FAILED - TAMPERED!'}")

# ── File timeline reconstruction ──────────────────────────────────────────────
print("\n=== NTFS Timeline Artefacts (Simulated) ===")

artefacts = [
    ("2026-05-28 08:02:11", "LOGIN",    "Administrator logged on (Event 4624)"),
    ("2026-05-28 08:14:33", "DOWNLOAD", "7za.exe downloaded to C:\\Users\\admin\\Downloads"),
    ("2026-05-28 08:15:01", "EXEC",     "7za.exe executed (Prefetch: 7za.exe-DEADBEEF.pf)"),
    ("2026-05-28 08:15:10", "ARCHIVE",  "archive.7z created in C:\\Users\\admin\\Documents"),
    ("2026-05-28 08:17:44", "NETWORK",  "HTTPS connection to 198.51.100.42:443 (62 MB transferred)"),
    ("2026-05-28 08:19:01", "DELETE",   "archive.7z deleted (MFT entry 0x3A9C marked free)"),
    ("2026-05-28 08:19:45", "LOGOUT",   "Administrator logged off (Event 4634)"),
]

for ts, etype, desc in artefacts:
    print(f"  {ts}  [{etype:<8}]  {desc}")

print("\n  Narrative: Administrator downloaded and executed an archiver, created an archive,")
print("  transferred 62 MB externally over HTTPS, then deleted the archive. Exfiltration suspected.")

# ── Memory forensics: simulated process list ──────────────────────────────────
print("\n=== Simulated Volatility pslist Output ===")
processes = [
    ("System",       4,    0,  "SYSTEM",    False),
    ("lsass.exe",    668,  568,"SYSTEM",    False),
    ("svchost.exe",  900,  568,"NETWORK",   False),
    ("explorer.exe", 2340, 1200,"ADMIN",    False),
    ("powershell.exe",3100,2340,"ADMIN",    True),
    ("cmd.exe",      3220,3100,"ADMIN",     True),
    ("7za.exe",      4100,3220,"ADMIN",     True),
]

print(f"  {'Name':<20} {'PID':>6} {'PPID':>6} {'User':<10}  Flag")
print("  " + "-"*55)
for name, pid, ppid, user, suspicious in processes:
    flag = " <-- SUSPICIOUS (spawned by Explorer)" if suspicious else ""
    print(f"  {name:<20} {pid:>6} {ppid:>6} {user:<10} {flag}")


=== Evidence Integrity Verification ===
  Original image SHA-256 : a3170f0ac498b9d58f5c46ee1f356984b9300f6f4c374918d347508c8d013efa
  Re-verified SHA-256    : a3170f0ac498b9d58f5c46ee1f356984b9300f6f4c374918d347508c8d013efa
  Integrity check        : PASSED
  Tampered image SHA-256 : 2153377eee63fb0c748430ab848937cfd5b90f9d4e1ec33a50693de961ca1988
  Tampered check         : FAILED - TAMPERED!

=== NTFS Timeline Artefacts (Simulated) ===
  2026-05-28 08:02:11  [LOGIN   ]  Administrator logged on (Event 4624)
  2026-05-28 08:14:33  [DOWNLOAD]  7za.exe downloaded to C:\Users\admin\Downloads
  2026-05-28 08:15:01  [EXEC    ]  7za.exe executed (Prefetch: 7za.exe-DEADBEEF.pf)
  2026-05-28 08:15:10  [ARCHIVE ]  archive.7z created in C:\Users\admin\Documents
  2026-05-28 08:17:44  [NETWORK ]  HTTPS connection to 198.51.100.42:443 (62 MB transferred)
  2026-05-28 08:19:01  [DELETE  ]  archive.7z deleted (MFT entry 0x3A9C marked free)
  2026-05-28 08:19:45  [LOGOUT  ]  Administrator logged off (

## Review Questions (MCQ)

**Q1.** The chain of custody primarily ensures:
A. Evidence is encrypted  B. Any person who handled evidence is documented, supporting legal admissibility  C. The forensic image is compressed  D. The hard drive is wiped after analysis

**Q2.** A write blocker is used to:
A. Prevent the suspect from accessing the system  B. Prevent any write to the source device during acquisition  C. Encrypt the forensic image  D. Block network traffic during imaging

**Q3.** A physical forensic image differs from a logical image in that it:
A. Is faster to create  B. Includes unallocated space, deleted files, and slack space  C. Only copies active files  D. Is compressed

**Q4.** NTFS stores four timestamps per file. The acronym for this set is:
A. CRUD  B. MACB (Modified, Accessed, Changed, Born)  C. ACID  D. ITAR

**Q5.** Fileless malware specifically evades which forensic technique?
A. Memory forensics  B. Network forensics  C. Disk-based file carving and artefact recovery  D. Hash verification

**Q6.** Volatility's `malfind` plugin looks for:
A. Running processes  B. Network connections  C. Memory regions marked executable with suspicious content  D. Registry hives

**Q7.** Secure file deletion (overwriting before delete) defeats:
A. Memory forensics  B. File carving from unallocated space  C. Log analysis  D. Timestamp correlation

**Q8.** Windows Event ID 1102 indicates:
A. A new process created  B. The Security event log was cleared  C. A privilege escalation  D. A failed login

**Q9.** NetFlow differs from full packet capture in that NetFlow:
A. Captures payload content  B. Records only flow-level metadata (no payload)  C. Requires a write blocker  D. Only records TCP traffic

**Q10.** Which hash algorithm is preferred for new forensic investigations?
A. MD5  B. SHA-1  C. SHA-256  D. CRC-32

*Answers: Q1 B, Q2 B, Q3 B, Q4 B, Q5 C, Q6 C, Q7 B, Q8 B, Q9 B, Q10 C.*

## Lab Assignment

**Part A -- Image acquisition**: Create a test forensic image of a USB drive or a virtual disk using `dd` or FTK Imager. Compute SHA-256 before and after acquisition. Verify that the hashes match. Document the command used and the exact hash values.

**Part B -- File carving**: Using Autopsy or PhotoRec on the image from Part A (or a provided sample image), recover at least three deleted files. Document the file type, recovered content, and whether the file was recoverable because TRIM was not applied.

**Part C -- Timeline creation**: Using Autopsy's timeline feature (or `mactime` from The Sleuth Kit), extract a sorted timeline of file system artefacts from a test image. Identify a five-minute window of intense activity and describe what a forensic investigator would infer from it.

**Part D -- Memory analysis**: Obtain a publicly available Windows memory sample (from Volatility's test images). Run `pslist`, `netscan`, and `cmdline`. Identify any process that should not be running (unusual parent-child relationship, suspicious command line, or network connection to a non-standard port).

## References

```{bibliography}
:filter: docname in docnames
```

1. Zamil, M. Z. H., and Khan, T. M. (2025). AI-Driven Digital Evidence Triage in Digital Forensics: A Comprehensive Review. IEEE ISDFS 2025.
2. Mohamed, H., Koroniotis, N., Schiliro, F., and Moustafa, N. (2025). IoT-CAD: A Comprehensive Digital Forensics Dataset for AI-based Cyberattack Attribution Detection in IoT. Ad Hoc Networks 174, 103840.
3. Gopinath, A., Kukatlapalli, P. K., Saleem K. M., S., and John, J. (2023). Explainable IoT Forensics: Investigation on Digital Evidence. IEEE InC4 2023.
4. Djenna, A., Barka, E., Benchikh, A., and Khadir, K. (2023). Unmasking Cybercrime with Artificial-Intelligence-Driven Cybersecurity Analytics. Sensors 23(14), 6302.
5. NIST SP 800-86: Guide to Integrating Forensic Techniques into Incident Response.
6. ACPO Good Practice Guide for Digital Evidence; The Sleuth Kit/Autopsy; Volatility Foundation.


```{index} Order of volatility, Chain of custody, Write blocker / forensic image / hashing, Daubert standard / Federal Rules of Evidence, AI evidence triage, IoT forensics, Explainable AI, Cyberattack attribution detection, Forensic image, Write blocker, Hash verification, Deleted file recovery, Slack space, MFT, Memory forensics, Volatility, PCAP, Artefact, Anti-forensics
```
